# Exercise 0 — Create a SparkSession

**Worked solution** · [All exercises](../index.html) · [Setup](../README.md)

## What you’ll learn

- Create a local SparkSession and explain how it differs from the notebook’s Python kernel.
- Run a small DataFrame job, recognise session reuse, and stop Spark when finished.

**Core: about 5 minutes.** The same baseline for everyone.

Saved output labels the author's interpreter path as `<validation-python>`; running the cell yourself prints your actual path. Run cells in order and end with **Finish — stop Spark**. This exercise does not save pipeline functions.

## Setup — check the Python kernel

Complete the [installation and setup check](../README.md#2-create-the-virtual-environment) first. Select the lab's `.venv` kernel, then run this cell. On Windows the printed path should end in `labs\.venv\Scripts\python.exe`; on macOS/Linux, `labs/.venv/bin/python`. If it points at uv's base Python instead, use [kernel selection help](../README.md#4-open-exercise-0-and-select-the-kernel). Importing `SparkSession` makes the class available; this cell does not start Spark.

In [1]:
import os
import sys

from pyspark.sql import SparkSession

print("Notebook Python:", sys.executable)

Notebook Python: <validation-python>


<a id="exercise-0"></a>
## Your task

**Where does the `spark` in `spark.read.parquet(...)` come from?**

Your notebook kernel is a running Python process. Selecting that kernel makes the installed PySpark package available; it does not create a SparkSession. `SparkSession` is the entry point for creating DataFrames, reading data and running SQL. We conventionally name the session `spark`.

In this lab, Spark runs locally on your laptop. PySpark starts the Java-based Spark runtime when we create the session. This is why both Python and Java were needed during installation.

### Supplied local settings

Run this cell before starting Spark. `PYSPARK_PYTHON` tells Spark which Python to use for Python workers. `SPARK_LOCAL_IP` sets Spark’s local IP to the loopback address for this laptop exercise. Neither line starts Spark.

In [2]:
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"

### Your code — create the session

Build a session using the pieces below. Use two local worker threads, give the application a name of your choice, and assign the resulting session to `spark`.

| Piece | What it does |
|---|---|
| `SparkSession.builder` | Begins configuring a session. |
| `.master("local[2]")` | Runs Spark on this machine with two worker threads. This does not create two machines or two executors. |
| `.appName("...")` | Gives the Spark application a recognisable name. |
| `.getOrCreate()` | Returns an existing session, or starts one when none exists. |

Chain these calls together. The builder configures; `getOrCreate()` gives you the session. See the [builder documentation](https://spark.apache.org/docs/4.2.0/api/python/reference/pyspark.sql/api/pyspark.sql.SparkSession.builder.getOrCreate.html) and [local execution settings](https://spark.apache.org/docs/4.2.0/submitting-applications.html#master-urls).

In [3]:
spark = SparkSession.builder.master("local[2]").appName("My first SparkSession").getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/22 19:33:22 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


<details>
<summary>Need a nudge?</summary>

Start from `SparkSession.builder`. Each configuration method returns the builder, so you can continue the chain with the next method. Finish with the method that returns the session. Parentheses let you put the chain on several lines.

</details>

### Check — ask Spark to do some work

The next cell checks your session and runs a tiny DataFrame. `range(3)` describes rows with IDs 0, 1 and 2; `show()` asks Spark to compute and display them. `count()` returns the number of rows to Python. Creating a session alone does not run our sales pipeline.

The first startup can take a little time. Wait for the cell to finish. Java or connection error? Use [setup troubleshooting](../README.md#2-create-the-virtual-environment); a running notebook kernel alone does not prove Spark is ready.

In [4]:
assert isinstance(spark, SparkSession), "Create spark with your builder expression first."
assert spark.sparkContext.master == "local[2]", 'Use .master("local[2]") for this exercise.'
print(f"Spark {spark.version}; application: {spark.sparkContext.appName}")
spark.sparkContext.setLogLevel("ERROR")
example = spark.range(3)
example.show()
assert example.count() == 3
print("Session check passed")

Spark 4.2.0; application: My first SparkSession


+---+
| id|
+---+
|  0|
|  1|
|  2|
+---+

Session check passed


### Predict, then run

If we call `getOrCreate()` again, do we get a second running Spark application? Predict the result of `same_session is spark` before running this supplied cell.

In [5]:
same_session = SparkSession.builder.getOrCreate()
print("Same session:", same_session is spark)
assert same_session is spark

Same session: True


### Why the next notebooks use `create_spark`

From Exercise 1 onward, the supplied Setup cell calls `spark = create_spark(RUN_ROOT)`. This is a helper written for this lab, not a PySpark API. It uses the same builder pattern you just used, then applies the lab defaults: local threads, Python and networking settings, UTC timestamps, two shuffle partitions and a per-run warehouse directory. It also checks the installed versions and reduces log noise.

You can read the helper in [workshop_runtime.py](../workshop_runtime.py). It returns a normal `SparkSession`. Later exercises use it so you can concentrate on the data; they start their own sessions and load your saved transformation functions. No live session is passed between notebooks.

<a id="finish"></a>
## Finish — stop Spark

Run the cell below after the checks. In this local lab, `spark.stop()` stops the underlying SparkContext and releases its resources. The Python kernel keeps running, but the stopped session and its DataFrames cannot run more work. To repeat this exercise, run the builder and following cells again. [SparkSession.stop documentation](https://spark.apache.org/docs/4.2.0/api/python/reference/pyspark.sql/api/pyspark.sql.SparkSession.stop.html).

In [6]:
spark.stop()
print("Spark stopped; the notebook Python kernel is still running.")

Spark stopped; the notebook Python kernel is still running.


Next: [Exercise 1 — Inspect the inputs](01-inspect.ipynb).